# Experiment 4: Experiment Tracking and Reproducibility Workflows

### AIM
Implement ML experiment tracking and reproducibility workflows using **MLflow** by recording parameters, metrics, preprocessing configurations, datasets, and trained artifacts. Compare multiple experiment runs, analyze the effects of preprocessing, hyperparameter tuning, and feature selection, and maintain experiment lineage and reproducibility across repeated executions.

---
### Workflow Architecture:
1. **Environment Setup & MLflow Configuration**: Connect to SQLite backend `sqlite:///mlflow.db`.
2. **Dataset Loading & Lineage Tracking**: Load `News_Category_Dataset_v3.json` with deterministic sampling and SHA-256 fingerprinting.
3. **Four Systematic Model Runs**:
   - **Run 1: MultinomialNB Baseline** (Standard Unigram TF-IDF, 10,000 features)
   - **Run 2: Logistic Regression with Feature Engineering** (N-grams (1,2), sublinear TF scaling, 25,000 features)
   - **Run 3: Hyperparameter Tuning & Class Balancing** (C=2.0, class_weight='balanced')
   - **Run 4: Linear Support Vector Classifier** (High-margin LinearSVC)
4. **Diagnostic Logging**: Confusion matrices, classification reports, and serialized models.
5. **Run Comparison & Analysis**: Automated comparison matrix across test accuracy, F1-scores, and latency.
6. **Reproducibility Test**: Deterministic execution verification with fixed random seeds.


In [ ]:
import sys
import os
from pathlib import Path

# Set working directory to project root
PROJECT_ROOT = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f"Working Directory set to: {PROJECT_ROOT}")

## 1. Initialize MLflow Tracking Client & Database

In [ ]:
import mlflow
from src.config import MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT_NAME

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print(f"MLflow Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"MLflow Experiment:   '{MLFLOW_EXPERIMENT_NAME}'")

## 2. Load Dataset & Track Dataset Lineage

In [ ]:
from src.data_loader import load_raw_dataframe, prepare_train_test_split, get_data_lineage_metadata

# Load dataset (sampled to 25,000 for rapid interactive execution; toggle in src/config.py)
df = load_raw_dataframe()
lineage_meta = get_data_lineage_metadata(df)
print("Dataset Lineage Metadata:")
for k, v in lineage_meta.items():
    print(f"  {k}: {v}")

X_train_raw, X_test_raw, y_train, y_test = prepare_train_test_split(df)
print(f"Training Samples: {len(X_train_raw)} | Testing Samples: {len(X_test_raw)}")

## 3. Execute Systematic Experiment Runs with MLflow Tracking

In [ ]:
from src.train_mlflow_exp4 import run_experiment_4_suite

# Executes the 4 systematic runs, logs all metrics, plots, and models to MLflow
comparison_df = run_experiment_4_suite()

## 4. Run Comparison Table & Analysis

In [ ]:
import pandas as pd
from IPython.display import display

cols = ["Run", "Model", "test_accuracy", "f1_weighted", "precision_weighted", "recall_weighted", "total_latency_seconds"]
display(comparison_df[cols].style.highlight_max(subset=["test_accuracy", "f1_weighted"], color="lightgreen"))

## 5. Visualizing Saved Diagnostic Plots

In [ ]:
from IPython.display import Image, display

cm_files = [
    "artifacts/cm_multinomialnb_baseline.png",
    "artifacts/cm_logisticregression_advanced_tfidf.png",
    "artifacts/cm_linearsvc_classifier.png"
]

for cm in cm_files:
    if os.path.exists(cm):
        print(f"Displaying: {cm}")
        display(Image(filename=cm))

## 6. Launch MLflow Interactive Web UI
To inspect the interactive MLflow UI directly in your browser or VS Code simple browser, open a terminal in VS Code and run:
```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```
Navigate to `http://127.0.0.1:5000` to visualize run comparisons, parameter scatter plots, metric progressions, and downloadable artifacts.
